Check before running:
1. Check api key is correctly named and stored
2. Check if the examples DMN XML file uploaded to computer with correct file name!
2. check the model name
3. Check the TEMPERATURE
4. Check max tokens
5. Paste the description
6. Change the ID of description

In [ ]:
# experiment_runner_mistral_zero_shot.py

# ── 0. Setup ───────────────────────────────────────────────
!pip install -q mistralai pandas

import os

from mistralai.client import Mistral

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-api==1.38.0, but you have opentelemetry-api 1.39.1 which is incompatible.
opentelemetry-sdk 1.38.0 requires opentelemetry-semantic-conventions==0.59b0, but you have opentelemetry-semantic-conventions 0.60b1 which is incompatible.


In [ ]:
from google.colab import files
from google.colab import userdata
import pandas as pd
import os
import re

os.makedirs("examples", exist_ok=True)
os.makedirs("experiments/dmn", exist_ok=True)

In [ ]:
# ── 1. Upload example DMN files ────────────────────────────

uploaded = files.upload()

for file_name in uploaded.keys():
    os.rename(file_name, f"examples/{file_name}")

print("Uploaded files:", os.listdir("examples"))

# ── 2. Load and validate example files ─────────────────────
def load_example(file_path: str) -> str:
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Missing file: {file_path}\n"
            "Make sure you uploaded all required files."
        )

    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

example_1_xml = load_example("examples/example_1.dmn")

Saving example_1.dmn to example_1.dmn
Uploaded files: ['example_1.dmn']


In [ ]:
# ── 3. Insert textual descriptions manually ────────────────
example_1_text = """An institution decides to distribute scholarships but can obviously not give them to everyone. Therefore, they decide to distribute them based on the grades, annual income and whether that person already received any scholarships for the year they are applying to. In short a person can only be eligible for a scholarship if the grades are excellent or good, they earn less than 50000 a year and have not received any other scholarship yet. All the other cases make that the person is not eligible for that scholarship."""

# ── 4. API key and model ───────────────────────────────────

#Name the API Key in the secrets the same way
API_KEY = userdata.get("MISTRAL_API_KEY")

client = Mistral(api_key=API_KEY)

#Check the model
MODEL_NAME = "mistral-large-2512"


In [ ]:
# ── 5. Configuration ───────────────────────────────────────
# Change this manually for each temperature run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 3
MAX_TOKENS = 8500
TOP_P = 1.0

# ── 6. New description for generation ──────────────────────
#Change the ID of the description based
description_id = "description_1"

description = """The BMI of a person can be calculated based on weight in kgs and length of a person in meters by using the following formula: weight/(length*length). If the BMI value is above 30, then the BMI-level is considered Obese. If you are a male and the BMI-value is under 18.5 then the BMI-level is severely underweight and if you are female, then you are considered underweight with the same bmi-value. If the BMI-value is between 18.5 and 25 (without 25), then the BMI-level is underweight for a male and normal for a female. Lastly, if the BMI-value is between 25 and 30 and you are a Male then the BMI-level is normal but if you are a female then BMI-level is overweight."""


# ── 7. Prompt builder ──────────────────────────────────────
def build_few_shot_prompt(description: str) -> str:
    return f"""<s>[INST] You are an expert in generating DMN 1.3 XML files compatible with Camunda Modeler.(Persona)

Use the following Camunda-exported DMN example as the structural reference.(Instruction)

Example description: (Context)
{example_1_text}

Example DMN XML:
{example_1_xml}

Generate a complete Camunda-compatible DMN XML file for the new description below.

Requirements:(Constraint instruction)
- Return ONLY raw XML.
- Do not include explanations, comments, headings, labels, markdown, or code fences.
- Start with:
  <?xml version="1.0" encoding="UTF-8"?>
- End with:
  </definitions>

- Include valid DMN 1.3 namespace declarations.
- Include inputData, decision, decisionTable, informationRequirement, and DMNDI elements.
- Every decision must contain exactly one decisionTable.
- Every decisionTable must use hitPolicy="UNIQUE".
- Rules must be mutually exclusive so that no input combination matches more than one rule.
- Every id attribute must be globally unique across the entire XML document.
- Never reuse the same id value in different elements.

- Use <inputEntry><text>-</text></inputEntry> to represent any value ("don't care").
- Escape XML special characters correctly: &amp; &lt; &gt;.
- Do not use raw "<" inside XML text nodes.
- Use &lt; for less-than comparisons inside <text> elements.
- Do not use undeclared XML entities such as &ge; or &le;.

- Ensure all XML tags are correctly opened and closed.
- Never leave incomplete XML elements.
- Every dmndi:DMNShape must close with </dmndi:DMNShape>.
- Every dmndi:DMNEdge must close with </dmndi:DMNEdge>.

- Follow the XML structure, namespace style, element order, decision table style, and DMNDI style of the example as closely as possible.
- Keep the DMNDI layout simple and valid.
- The file must be importable and readable in Camunda Modeler.

Text: '{description}' [/INST]</s>"""

# ── 8. Clean model output ──────────────────────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()

# ── 9. Run generation ──────────────────────────────────────
prompt = build_few_shot_prompt(description)

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Mistral | {description_id} | few_shot | temp={TEMPERATURE} | iter={iteration}")

    response = client.chat.complete(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        top_p=TOP_P,
    )

    usage = response.usage

    print("Input tokens:", usage.prompt_tokens)
    print("Output tokens:", usage.completion_tokens)
    print("Total tokens:", usage.total_tokens)

    dmn_xml = clean_model_output(response.choices[0].message.content)

    base_name = f"{description_id}_mistral_few_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    # Save DMN file
    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

# ── 10. Download DMN results ───────────────────────────────
for iteration in range(1, N_ITERATIONS + 1):
    base_name = f"{description_id}_mistral_few_shot_temp_{TEMPERATURE}_iter_{iteration}"
    files.download(f"experiments/dmn/{base_name}.dmn")

▶ Mistral | description_1 | few_shot | temp=0.6 | iter=1
Input tokens: 3226
Output tokens: 2202
Total tokens: 5428
Saved: experiments/dmn/description_1_mistral_few_shot_temp_0.6_iter_1.dmn


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>